# 01 - Scraping & Tokenization
Scrape reviews from Google Play Store (M-Tix / Cinema 21), then perform:
- Tokenization
- Lowercasing
- Punctuation removal

**App ID:** `lds.cinema21`

In [ ]:
# Install dependencies
!pip install google-play-scraper PySastrawi

In [ ]:
import pandas as pd
import re
import time
import nltk
import matplotlib.pyplot as plt
from google_play_scraper import reviews_all, Sort
from nltk.tokenize import word_tokenize

nltk.download('punkt')
nltk.download('punkt_tab')

## 1. Scraping

In [ ]:
def scrape_reviews(app_id, langs, country='id'):
 all_dfs = []
 for lang in langs:
 print(f'Scraping lang={lang}, country={country}...')
 start = time.time()
 result = reviews_all(
 app_id,
 lang=lang,
 country=country,
 sort=Sort.NEWEST,
 sleep_milliseconds=0
 )
 elapsed = time.time() - start
 print(f'Done: {len(result)} reviews in {elapsed:.2f}s')
 df_lang = pd.DataFrame(result)
 df_lang['lang'] = lang
 all_dfs.append(df_lang)
 df_all = pd.concat(all_dfs, ignore_index=True)
 print(f'Total: {len(df_all)} reviews')
 return df_all

df_raw = scrape_reviews('lds.cinema21', langs=['id', 'en'])
df_raw = df_raw[['content', 'score']].dropna(subset=['content'])
df_raw = df_raw[df_raw['content'].str.strip() != '']
print(df_raw.shape)
df_raw.head()

## 2. Save Raw Data

In [ ]:
df_raw.to_csv('mtix_raw.csv', index=False)
print(f'Raw data saved: {df_raw.shape[0]} rows')

## 3. Tokenization

In [ ]:
df_raw['tokenized'] = df_raw['content'].apply(word_tokenize)
df_raw[['content', 'tokenized']].head()

## 4. Lowercasing

In [ ]:
df_raw['lower_tokens'] = df_raw['tokenized'].apply(
 lambda tokens: [t.lower() for t in tokens]
)
df_raw[['tokenized', 'lower_tokens']].head()

## 5. Punctuation Removal

In [ ]:
def remove_punctuation(tokens):
 return [t for t in tokens if re.match(r'[a-zA-Z0-9]', t)]

df_raw['clean_tokens'] = df_raw['lower_tokens'].apply(remove_punctuation)
df_raw[['lower_tokens', 'clean_tokens']].head()

## 6. Save Tokenized Data

In [ ]:
df_raw.to_csv('mtix_tokenized.csv', index=False)
print('Tokenized data saved.')
df_raw[['content', 'score', 'clean_tokens']].head(10)